# Kineret — Complication Prediction

**Task.** Observe the first **K days** of an admission; predict, per
complication, whether it occurs before day 14, **when**, and how long the stay
will be.

```text
inputs :  events in [0, K*24]                hours from ADMISSION
labels :  outcome occurs in (K*24, 14*24]    hours from ADMISSION
```

**K is a training-time augmentation, not a sweep.** Each training admission
contributes one sample per context window it reaches — the same trajectory cut
at a different point, with the label window re-derived to match — so the model
learns to forecast from however much history it has. The augmentation is
deliberately **ragged**: a patient discharged on day 5 has no day-7 cut to make,
so long stays contribute more samples than short ones. Cohort membership keys on
the *evaluation* window only, so widening the augmentation adds samples and never
removes patients.

**Evaluation happens at a single window**, chosen in §2 from the support table
before any training starts. Every reported number comes from that one window, so
the results are a table, not a surface.

## The seven arms

| Rung | Arm | Input |
|---|---|---|
| 1 | LogReg (raw) | raw measurements, distribution-summarised |
| 2 | LogReg (raw + QA) | + aggregated compliance in the context vector |
| 3 | ss-STraTS (raw) | raw measurements + learned temporal attention |
| 4 | ss-STraTS (raw + QA) | + aggregated compliance in the context vector |
| 5 | INTERVenE-Enc (σ-bins) | knowledge-**free** intervals over the same raw data |
| 6 | INTERVenE-Enc (KB) | knowledge-**based** Mediator abstractions |
| 7 | INTERVenE-Enc (KB + QA) | + treatment-quality pattern tokens |

Rungs 5 → 6 isolate the knowledge base. Rungs 6 → 7 isolate the treatment
signal. Rungs 2/4 ask the same QA question of the models that cannot represent
pattern tokens, so the QA effect is separable from the architecture.

Every arm answers the same three questions, on the same patients, with the same
labels, over the same reconciled event set.

---

**Run order.** §1 config → §2 support and window choice → §3 wiring check →
§4 the ladder. §5 onward reads from `outputs/` and can be re-run freely.

**Resume.** §4 skips any arm that already has a `test_predictions.csv`. If the
kernel dies, re-run that one cell and it continues from where it stopped.

**Memory.** Each arm is trained, scored, written to disk and released. Nothing
is held across cells.

**Figures.** Every figure and its underlying table are written to
`outputs/figures/` as PNG + PDF + CSV, ready for the paper.

## 1 · Study configuration

**This cell is the experimental design.** Everything shared between models is
set here and nowhere else — windows, targets, date range, event reconciliation,
QA aggregation, seeds. Per-model hyperparameters stay with their model
(`kineret/strats/config.py`, `kineret/intervene/config/model_config.py`,
`kineret/logreg/train.py`).

`TARGETS` is the study's target list. It ships as every clinical complication
the Mediator knowledge base can emit, minus structural framing and treatment
administrations — **trim it to the paper's list here.** §2 shows which entries
clear the support floor; only those get a prediction head.

In [ ]:
import gc, os, sys, warnings
sys.path.insert(0, os.path.abspath(".."))
warnings.filterwarnings("ignore", category=FutureWarning)

# --- Synthetic data (optional) -------------------------------------------
# Uncomment to exercise the whole pipeline without the real Kineret drop.
# !python ../scripts/make_synthetic_data.py --patients 400 --out ../data/source_synth
# for key, nm in [("RAW_TEMPORAL", "mediator_input"), ("ABSTRACT", "mediator_output"),
#                 ("CONTEXT", "context_data"), ("QA", "qa_scores")]:
#     os.environ[f"KINERET_{key}"] = os.path.abspath(f"../data/source_synth/{nm}.csv")

from kineret.config import data_config as C
from kineret.config import paths

# ===========================================================================
#  THE STUDY DESIGN -- edit here, nowhere else
# ===========================================================================
TARGETS = [
    "CARDIO-VASCULAR_DISORDER_EVENT",
    "DEATH_EVENT",
    "HYPERGLYCEMIA_EVENT",
    "HYPEROSMOLALITY_EVENT",
    "HYPOGLYCEMIA_EVENT",
    "INFECTION_EVENT",
    "KETOACIDOSIS_EVENT",
    "KIDNEY_COMPLICATION_EVENT",
    "SEVERE_HYPERGLYCEMIA_EVENT",
    "SEVERE_HYPOGLYCEMIA_EVENT",
]

# ===========================================================================
#  RAW-SIDE EVENT DERIVATION
# ===========================================================================
# mediator_input.csv holds measurements, not complications: the Mediator is what
# turns a glucose reading into HYPERGLYCEMIA_EVENT. Without the raw file's own
# view of those events the two files have nothing to agree on, and the
# cross-file intersection is a no-op for exactly the targets that carry the
# study.
#
# The rules are NOT written here. They are compiled from the Mediator's own
# knowledge base and executed verbatim, because the Mediator produced
# mediator_output.csv FROM mediator_input.csv -- so any disagreement is a
# reproduction defect, never a source conflict, and section 2 refuses to
# continue on one.
#
#   python -m kineret.mediator_rules <path-to>/Mediator/core/knowledge-base
#
# ships kineret/config/event_rules.json. Point MEDIATOR_KB_PATH at a live
# checkout to compile from the XML at run time instead -- the safer choice if
# the knowledge base has moved on since this package was built.
MEDIATOR_KB_PATH = None      # e.g. "/workspace/Mediator/core/knowledge-base"

study = C.configure_study(
    # --- windows ---------------------------------------------------------
    # Training augmentation: one sample per window each patient reaches.
    # The range spans the whole horizon -- every window is a real cut of data
    # already collected, and membership does not depend on it, so a wide range
    # costs nothing and buys samples.
    train_context_days=list(range(1, 14)),   # 1..13
    eval_context_days=4,                     # single; revisit after §2
    horizon_end_days=14.0,

    # K sets both how much history a sample gets and how far ahead it forecasts.
    # Hand both to the model rather than making it infer them.
    add_context_length_features=True,

    # --- cohort ----------------------------------------------------------
    date_range_start="2022-07-01",
    date_range_end="2025-12-31",
    date_range_require_full_horizon=True,

    # --- targets ---------------------------------------------------------
    targets=TARGETS,
    auto_discover_outcomes=False,            # the list above is the list
    outcome_support_threshold=0.01,          # 1 % of train patients

    # Every OTHER `*_EVENT` the Mediator emits (AKI, ELECTROLYTE_DERANGEMENT,
    # MYOCARDIAL_INJURY, ...) is a KB conclusion, not a target. It stays in the
    # KB arms' input and is stripped from everyone else's: handing a derived
    # clinical judgement to LogReg or ss-STraTS would make the ladder measure a
    # leak instead of the value of abstraction.
    kb_events_for_kb_arms_only=True,

    # --- shared data contract -------------------------------------------
    event_source="mediator",                 # whose event verdict wins
    event_match_tolerance_h=24.0,            # clinical: same event restated

    # For a target BOTH files carry, credit only the occurrences they date
    # identically -- this is what removes the 3096-vs-3098 support drift.
    # Targets only one file carries (the Mediator-derived ones) pass through.
    event_align_across_files=True,
    event_align_tolerance_min=1.0,           # clock skew between two exports

    # Rebuild the raw file's own view of every complication FIRST, so the
    # intersection above has something to intersect for all ten targets.
    derive_raw_events=True,
    mediator_kb_path=MEDIATOR_KB_PATH,
    # The derived view must reproduce the Mediator exactly. Anything less means
    # a rule is being executed wrongly, and the intersection would silently
    # delete real labels -- so the build stops instead.
    rule_agreement_min=1.0,
    raise_on_rule_mismatch=True,
    events_as_inputs=True,                   # in-window events are history
    qa_aggregations=["mean"],                 # compliance -> context vector

    # --- identity --------------------------------------------------------
    # The Mediator carries one id column, so the temporal tables key on the
    # ADMISSION. The context table keys on VisitId and names the person too.
    context_id_column="VisitId",
    context_person_column="person_id",
    # One person's admissions never straddle the split -- otherwise the test
    # score partly measures having memorised that person from training.
    split_group_by_person=True,

    # --- reproducibility -------------------------------------------------
    split_seed=2023,
    seed=2023,
)

BOOTSTRAP = 2000        # resamples for every reported interval
print(C.study_summary())

In [ ]:
from kineret.benchmark import ARMS, resolve_device, free_memory, save_table

DEVICE = resolve_device()          # warns loudly if this lands on CPU
print()
for key, model, use_qa, label in ARMS:
    print(f"  rung {ARMS.index((key, model, use_qa, label)) + 1}  "
          f"{label:<32} model={model:<18} qa={use_qa}")
print()
print("Source tables (drop them in data/source/ under these exact names):")
print(paths.describe_sources())

## 2 · Compile the rules from *your* knowledge base

`kineret/config/event_rules.json` ships pre-compiled, but from whichever
Mediator checkout this package was built against — which may not be the one
that produced your `mediator_output.csv`. Point this at the knowledge base
that actually ran and recompile, so the reproduction is checked against the
right definitions.

Set `MEDIATOR_KB` to the absolute path of the `knowledge-base` directory
(the one containing `events/`, `raw-concepts/`, `states/`, …). Leave it
`None` to keep the shipped rules.

Read the printed rules before moving on: they are the definitions every
label in this study depends on.


In [ ]:
# ---------------------------------------------------------------------
#  Absolute path to the Mediator's knowledge base, or None for the
#  rules that ship with this package.
#     e.g. "/workspace/notebooks/code/Mediator/core/knowledge-base"
# ---------------------------------------------------------------------
MEDIATOR_KB = None

import json as _json, os as _os
from kineret import raw_events as _raw_events
from kineret.mediator_rules import parse_knowledge_base

if MEDIATOR_KB:
    kb = _os.path.abspath(_os.path.expanduser(MEDIATOR_KB))
    # Accept a repo root or a core/ directory too, so a slightly-off path is
    # corrected rather than compiling to an empty rule set.
    for candidate in (kb,
                      _os.path.join(kb, "knowledge-base"),
                      _os.path.join(kb, "core", "knowledge-base")):
        if _os.path.isdir(_os.path.join(candidate, "events")):
            kb = candidate
            break
    else:
        raise FileNotFoundError(
            f"No 'events/' directory under {kb}. Point MEDIATOR_KB at the "
            f"knowledge-base folder itself, e.g. "
            f".../Mediator/core/knowledge-base")

    compiled = parse_knowledge_base(kb)
    out_path = _raw_events.rules_path()
    with open(out_path, "w", encoding="utf-8") as handle:
        _json.dump(compiled, handle, indent=1, sort_keys=True)
    _raw_events._RULES_CACHE.clear()      # so the next cell reads the new file
    print(f"[rules] compiled from {kb}")
    print(f"[rules] {len(compiled['events'])} events, "
          f"{len(compiled['clippers'])} concept ranges -> {out_path}\n")
else:
    compiled = _raw_events.load_rules()
    print(f"[rules] using the rules shipped with the package "
          f"({_raw_events.rules_path()})\n")

# The definitions every label depends on -- read them.
for _name in TARGETS:
    _spec = compiled["events"].get(_name)
    if _spec is None:
        print(f"  {_name:<32} NO RULE -- this target cannot be reproduced")
        continue
    for _rule in _spec["rules"]:
        _parts = []
        for _clause in _rule["clauses"]:
            _kind = (_clause["transform"] or {}).get("kind", "value")
            _bounds = ", ".join(
                (f"{_c['type']}={_c['value']}" if "value" in _c
                 else f"range=[{_c['min']}, {_c['max']}]")
                for _c in _clause["constraints"])
            _parts.append(f"{_clause['source']}[{_kind}] {_bounds}")
        print(f"  {_name:<32} {_rule['operator']}( " + "  |  ".join(_parts) + " )")
        if _rule.get("unsupported"):
            print(f"      UNSUPPORTED: {_rule['unsupported']} -- section 3 "
                  f"will refuse to build")

_missing_ranges = [c for c in ("GLUCOSE_MEASURE", "CREATININE_SERUM_MEASURE")
                   if not compiled["clippers"].get(c)]
if _missing_ranges:
    print(f"\n[rules] WARNING: no validity range for {_missing_ranges}; "
          f"out-of-range readings will reach the rules.")


## 3 · Rule check — before anything expensive

The Mediator built `mediator_output.csv` from `mediator_input.csv`. This
cell re-executes its own compiled rules on a sample of admissions and
checks the result matches, occurrence for occurrence.

**Every target must read 100%.** `raw_only` events were invented by the
reproduction; `med_only` events were missed. Either one means a rule is
being executed wrongly, and the cross-file intersection later would delete
real labels rather than filter doubtful ones — so section 4 refuses to
build the cohort until this is clean.

Runs on a sample so a wrong rule surfaces in under a minute instead of
after a full cohort build.


In [ ]:
from kineret.raw_events import validate_rules

# n_admissions=0 checks every admission; a sample is enough to catch a
# wrong threshold and is much faster on the full extract.
rule_report = validate_rules(n_admissions=5000)
save_table(rule_report, "rule_agreement")

assert (rule_report["agreement"] >= 1.0).all(), (
    "Some targets do not reproduce the Mediator exactly -- see the table "
    "above. Fix the rule before continuing; do not relax the threshold.")
free_memory()


In [ ]:
import json, pandas as pd
from kineret.config import paths

taks = set(json.load(open(paths.TAK_REPO_PATH, encoding="utf-8"))["taks"])
abstract = pd.read_csv(paths.ABSTRACT_FILE, usecols=["ConceptName"])
missing = sorted(set(abstract["ConceptName"]) - taks)
print(f"TAK repo: {len(taks)} concepts | mediator_output: {abstract['ConceptName'].nunique()} distinct")
print("missing from TAK:", missing or "none")

## 4 · Target support — choosing the evaluation window

**Run this before training.** It builds the cohort and shows, for every
candidate window, how many positives each target still has in that window's
label range `(K, 14]`.

The trade-off is direct: a larger K gives the model more history but leaves a
shorter label window, and the rare complications lose positives first. The
number that matters is how many targets still clear the support floor — those
are the ones that get a prediction head.

The reconciliation audit below it is the other thing to check: the raw file and
the Mediator output disagree about how often each complication fired, and this
is where that is resolved into one canonical event set used by every arm.

In [ ]:
import pandas as pd
from kineret.benchmark import ensure_prepared, target_support
from kineret import figures

cohort = ensure_prepared()
support = target_support(cohort)

print(f"\nCohort: {len(cohort.patients):,} patients")
samples = cohort.samples()
for split in ("train", "val", "test"):
    rows = samples[samples["split"] == split]
    print(f"  {split:<5}: {len(rows):>6,} samples over "
          f"{rows['PatientId'].nunique():>5,} patients "
          f"(K={sorted(rows['k'].unique())})")
print(f"\nTargets with a head at K={C.EVAL_CONTEXT_DAYS}: "
      f"{len(cohort.outcome_names)} of {len(TARGETS)}")
print("  kept    : " + ", ".join(cohort.outcome_names))
if cohort.dropped_outcomes:
    print("  dropped : " + ", ".join(
        f"{k} ({v:.2%})" for k, v in sorted(cohort.dropped_outcomes.items(),
                                            key=lambda kv: -kv[1])))

In [ ]:
figures.plot_target_support(support);

In [ ]:
print("Event reconciliation -- the two source files, and the canonical result:")
save_table(cohort.event_audit, "cohort_event_audit")
cohort.event_audit

### Confirm the window

If §2 says a different K is the right call, change `eval_context_days` in §1 and
re-run from there. The cohort and the support filter both depend on it, so it
has to be settled before §4.

In [ ]:
print(f"Evaluating at K={C.EVAL_CONTEXT_DAYS}: observe days 0-{C.EVAL_CONTEXT_DAYS}, "
      f"predict days {C.EVAL_CONTEXT_DAYS}-{C.HORIZON_END_DAYS:.0f}.")
print(f"Training augmented over K={list(C.TRAIN_CONTEXT_DAYS)}.")
free_memory()

## 5 · Wiring check

Runs **all seven arms** with deliberately crippled hyperparameters. Minutes, not
hours. It proves four things:

1. every arm trains, predicts and scores without crashing;
2. the cohort, the ss-STraTS pickles and the σ-bin interval table all build;
3. **every arm is scored on byte-identical labels for identical test patients** —
   the invariant that makes §5 a comparison rather than seven unrelated numbers;
4. the date range and event reconciliation applied cleanly.

Metrics from this cell are meaningless and are not reported. It writes to a
throwaway directory and deletes it.

In [ ]:
from kineret.benchmark import smoke_test

smoke_report = smoke_test(device=DEVICE)
free_memory()
smoke_report[["label", "status", "seconds"]]

## 6 · Train the ladder — one cell

All seven arms, trained and scored end to end. This is the expensive cell and
the only one that trains anything.

* **Resumable** — arms with existing predictions are skipped. Delete an arm's
  directory under `outputs/` to force a retrain.
* **Memory-bounded** — each arm is released after scoring.
* **Fault-tolerant** — a failed arm is logged and the run continues; check the
  `status` column.
* Progress is written to `outputs/run_ladder_log.csv` after every arm.

In [ ]:
from kineret.benchmark import run_ladder

ladder_report = run_ladder(
    arms=None,                    # None = all seven, in ladder order
    device=DEVICE,
    bootstrap_resamples=BOOTSTRAP,
    skip_existing=True,           # <- the resume switch
)
save_table(ladder_report, "run_ladder_log")
free_memory(verbose=True)
ladder_report[["label", "status", "minutes"]]

## 7 · The ladder

The study's central claim, as one table and one figure:

> raw measurements < knowledge-free intervals < KB abstractions < KB + treatment patterns

Every number carries a 95 % interval from a 2 000-resample **patient-level
bootstrap** on the held-out split. **Adjacent rungs whose intervals overlap have
not been separated by this data** — the figure is a dot-and-interval plot rather
than a bar chart for exactly that reason.

In [ ]:
from kineret.benchmark import check_label_agreement, ladder_table, load_results

# Re-assert the invariant on the REAL runs, not just the smoke test.
agreement = check_label_agreement()
assert agreement["labels_identical"], agreement["detail"]

ladder = ladder_table(average="weighted")
save_table(ladder, f"ladder_weighted_k{C.EVAL_CONTEXT_DAYS}")
ladder

In [ ]:
figures.plot_ladder(average="weighted");

In [ ]:
# Macro average: every target weighted equally, so the rare complications count
# as much as the common ones. Weighted and macro disagreeing tells you the arms
# differ mostly on the rare tail.
macro = ladder_table(average="macro")
save_table(macro, f"ladder_macro_k{C.EVAL_CONTEXT_DAYS}")
macro

## 8 · The other two questions

Onset timing and length of stay. **Lower is better** in both panels, which is
why they are plotted apart from the risk metrics.

Onset is conditional on occurrence — only positive patients contribute — so on a
rare target the interval reflects which few patients were drawn rather than
model uncertainty. §7 reports `n_pos` and `ci_reliable` next to every figure.

In [ ]:
figures.plot_error_heads(average="weighted");

## 9 · Does the QA signal help?

Each model's two arms differ by **exactly** the aggregated `QA_<pattern>`
context block — same patients, same labels, same events, same seed — plus, for
INTERVenE only, the pattern token stream it can natively represent. So a
positive delta is attributable to the treatment-quality signal.

A delta narrower than the intervals in §5 is noise, not a finding.

In [ ]:
from kineret.benchmark import qa_delta_table

qa_delta = qa_delta_table(average="weighted")
save_table(qa_delta, "qa_delta")
qa_delta

In [ ]:
figures.plot_qa_delta(qa_delta);

## 10 · Per-target breakdown

Where does any advantage actually come from? `n_pos` is the positive support on
the held-out split and `ci_reliable` flags targets with too few positives for an
interval to mean much — treat a strong number on a single-digit-support target
as noise, not a result.

In [ ]:
from kineret.benchmark import per_outcome_table

per_auprc = per_outcome_table(metric="auprc")
save_table(per_auprc.reset_index(), "per_target_auprc")
per_auprc

In [ ]:
figures.plot_per_target(metric="auprc");

In [ ]:
figures.plot_per_target(metric="auroc");

## 11 — Fairness / subgroup analysis

Split the KB arm's held-out predictions by age band and sex and
report per-outcome AUROC / AUPRC inside each subgroup. Reads only
`test_predictions.csv` + the cohort's static context, so this cell
is post-hoc and does not re-train.

Note. The cohort's static context vector carries whichever
demographic columns were listed in `configure_study(...)`. If age /
sex are missing from `context_data.csv` in the closed environment,
this cell degrades to an empty table with a printed warning — no
downstream cell depends on its output.

In [ ]:
from kineret.evaluation import subgroup_metrics

cohort = ensure_prepared()
ctx    = cohort.context_data if hasattr(cohort, 'context_data') else None

if ctx is None or 'age' not in ctx.columns:
    print('[skip] age not in cohort.context_data — subgroup analysis unavailable')
else:
    # Age bands matching the cohort's typical stratification.
    ctx = ctx.copy()
    ctx['age_band'] = pd.cut(ctx['age'],
                              bins=[0, 55, 65, 75, 85, 120],
                              labels=['<55', '55-64', '65-74', '75-84', '85+'])

    from kineret.benchmark import run_dir_for
    kb_dir = run_dir_for('intervene_kb')

    for col in ('age_band', 'sex'):
        if col not in ctx.columns:
            print(f'[skip] {col!r} not in context'); continue
        df = subgroup_metrics(kb_dir, ctx, subgroup_col=col)
        save_table(df, f'subgroup_kb_{col}.csv')
        print(f'
=== {col} ===')
        print(df.pivot(index='outcome', columns='subgroup', values='auprc').round(3))

## 12 — Pairwise Δ with paired-bootstrap p-values + BH-FDR

For every headline arm contrast in the paper, compute a paired
patient-level bootstrap Δ CI and a two-sided p-value, then apply
Benjamini–Hochberg FDR control across the outcomes-per-comparison.
This complements the overlap-of-CIs criterion the ladder table already
reports.

In [ ]:
from kineret.evaluation import paired_bootstrap_delta, benjamini_hochberg
from kineret.benchmark   import run_dir_for

# The three contrasts the AIIM paper leans on.
CONTRASTS = [
    ('KB vs sigma-bins',     'intervene_kb',    'intervene_std'),
    ('KB vs ss-STraTS',      'intervene_kb',    'strats'),
    ('KB+QA vs KB',          'intervene_kb_qa', 'intervene_kb'),
]

rows = []
for name, arm_a, arm_b in CONTRASTS:
    a_dir = run_dir_for(arm_a); b_dir = run_dir_for(arm_b)
    for metric in ('auroc', 'auprc'):
        r = paired_bootstrap_delta(a_dir, b_dir, metric=metric,
                                     n_resamples=2000)
        rows.append({'contrast': name, 'metric': metric.upper(), **r})
delta_df = pd.DataFrame(rows)

# BH-FDR across the six (contrast, metric) tests.
bh = benjamini_hochberg(delta_df['p_value_two_sided'].to_numpy(), alpha=0.05)
delta_df['p_adj_BH'] = bh['p_adjusted']
delta_df['significant_q05'] = bh['reject']

save_table(delta_df, 'pairwise_delta.csv')
delta_df.round(4)

## 13 — Calibration summary

Per-arm expected calibration error, at 15 uniform bins, aggregated by
positive-count-weighted mean across outcomes. This is the compact
one-line summary; the full reliability curves and temperature-scaling
analysis live in `notebooks/calibration.ipynb`.

In [ ]:
from kineret.evaluation import (
    load_predictions, outcome_names_from, expected_calibration_error,
)

rows = []
for arm_key in ARMS:
    kb_dir = run_dir_for(arm_key)
    if not os.path.exists(os.path.join(kb_dir, 'test_predictions.csv')):
        continue
    preds = load_predictions(kb_dir)
    outs  = outcome_names_from(preds)
    per_outcome = []
    for o in outs:
        y = preds[f'label_{o}'].to_numpy(dtype=float)
        p = preds[f'prob_{o}' ].to_numpy(dtype=float)
        per_outcome.append((int((y == 1).sum()),
                            expected_calibration_error(y, p, n_bins=15)))
    wts = np.array([n for n, _ in per_outcome], dtype=float)
    eces = np.array([e for _, e in per_outcome], dtype=float)
    weighted = float((wts * eces).sum() / wts.sum()) if wts.sum() else float('nan')
    rows.append({'arm': arm_key, 'ECE_weighted_15bins': round(weighted, 4)})

calib_df = pd.DataFrame(rows)
save_table(calib_df, 'calibration_summary.csv')
calib_df

## 14 · Export

Everything for the paper is already in `outputs/figures/` — each figure as PNG
and PDF, with the table behind it as CSV.

In [ ]:
import json

results = load_results(average="weighted")
results.to_csv(os.path.join(paths.OUTPUT_DIR, "benchmark_summary.csv"), index=False)
with open(os.path.join(paths.OUTPUT_DIR, "study_config.json"), "w") as f:
    json.dump({k: (list(v) if isinstance(v, tuple) else v)
               for k, v in study.items()}, f, indent=2, default=str)

print(f"Artefacts in {paths.FIGURE_DIR}:")
for entry in sorted(os.listdir(paths.FIGURE_DIR)):
    print(f"  {entry}")
free_memory(verbose=True)
results